# 03 - QLoRA Fine-Tuning (Optimized)

Optimized configuration based on diagnostic results:
- **max_seq_length=512** (was 2048, avg token length is 107)
- **packing=True** for 5x speedup on short sequences
- **Logging to file** for real-time monitoring

Expected speedup: ~5-6x faster than baseline

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import os
import time
import json
import sys

os.chdir("/kaggle/working")
if not os.path.exists("fingpt-qlora"):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir("fingpt-qlora")

# Setup file logging
log_path = "results/training_log.txt"
os.makedirs("results", exist_ok=True)

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

log_file = open(log_path, "w")
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

def log(msg):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}")

log("Training started")

In [ ]:
log("=== Data Pipeline ===")
t0 = time.time()

if not os.path.exists("data/splits/train.json"):
    log("Running data pipeline...")
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits

t1 = time.time()
log(f"Data pipeline time: {t1-t0:.1f}s")

import json
with open("data/splits/train.json") as f:
    train_data = json.load(f)
with open("data/splits/val.json") as f:
    val_data = json.load(f)

log(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
import torch

log("=== Model Loading ===")
log(f"GPU: {torch.cuda.get_device_name(0)}")
log(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

t0 = time.time()

from unsloth import FastLanguageModel

# OPTIMIZED: Reduced from 2048 to 512 (avg token length is 107)
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512  # Was 2048, now 512 for 4x speedup

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

t1 = time.time()
log(f"Model loading time: {t1-t0:.1f}s")
log(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
log("=== LoRA Setup ===")
t0 = time.time()

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

t1 = time.time()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
log(f"LoRA setup time: {t1-t0:.1f}s")
log(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
log("=== Dataset Preparation ===")
t0 = time.time()

from datasets import Dataset

def load_split(path):
    with open(path) as f:
        data = json.load(f)
    texts = []
    for ex in data:
        messages = []
        for turn in ex["conversations"]:
            role = "assistant" if turn["from"] == "gpt" else turn["from"]
            messages.append({"role": role, "content": turn["value"]})
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append({"text": text})
    return Dataset.from_list(texts)

train_dataset = load_split("data/splits/train.json")
val_dataset = load_split("data/splits/val.json")

t1 = time.time()
log(f"Dataset prep time: {t1-t0:.1f}s")
log(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Verify token lengths
lengths = [tokenizer(train_dataset[i]["text"], return_tensors="pt")["input_ids"].shape[1] for i in range(min(100, len(train_dataset)))]
log(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")
log(f"Max seq length: {MAX_SEQ_LENGTH}")
log(f"Sequences fitting in max_length: {sum(1 for l in lengths if l <= MAX_SEQ_LENGTH)}/{len(lengths)}")

In [ ]:
log("=== Training Configuration ===")

from trl import SFTTrainer, SFTConfig

BATCH_SIZE = 2
GRAD_ACCUM = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4

# OPTIMIZED: Enable packing for 5x speedup
training_args = SFTConfig(
    output_dir="outputs/v1_optimized",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim="adamw_8bit",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,
    bf16=False,
    logging_steps=5,
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=100,
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=True,  # OPTIMIZED: Enable packing for 5x speedup
)

# Calculate expected steps
effective_batch = BATCH_SIZE * GRAD_ACCUM
total_steps = len(train_dataset) * NUM_EPOCHS // effective_batch

log(f"Effective batch size: {effective_batch}")
log(f"Total steps: {total_steps}")
log(f"Epochs: {NUM_EPOCHS}")
log(f"Learning rate: {LEARNING_RATE}")
log(f"Packing: True (expected 5x speedup)")
log(f"Max seq length: {MAX_SEQ_LENGTH} (reduced from 2048)")
log(f"Expected time: ~{total_steps * 2 / 60:.0f} minutes (est. 2s/step)")

In [ ]:
log("=== Starting Training ===")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

t0 = time.time()
log("Training started...")
trainer.train()
t1 = time.time()

total_time = t1 - t0
log(f"Training completed in {total_time:.0f}s ({total_time/60:.1f}min)")
log(f"Time per step: {total_time / trainer.state.global_step:.2f}s")

In [ ]:
import matplotlib.pyplot as plt

log("=== Plotting Training Curves ===")

os.makedirs("results/figures", exist_ok=True)

log_history = trainer.state.log_history
train_loss = [(l["step"], l["loss"]) for l in log_history if "loss" in l]
eval_loss = [(l["step"], l["eval_loss"]) for l in log_history if "eval_loss" in l]

fig, ax = plt.subplots(figsize=(10, 5))
if train_loss:
    steps, losses = zip(*train_loss)
    ax.plot(steps, losses, label="Train Loss", alpha=0.7)
if eval_loss:
    steps, losses = zip(*eval_loss)
    ax.plot(steps, losses, label="Eval Loss", marker="o", markersize=4)

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training Curves - v1_optimized")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/training_curve_v1_optimized.png", dpi=150)
plt.show()
log("Saved training curve")

In [ ]:
log("=== Generating Sample Outputs ===")

FastLanguageModel.for_inference(model)

SAMPLE_PROMPTS = [
    "Analyze the sentiment: Tesla shares surged 12% after record Q3 deliveries",
    "What are the key risks of investing in emerging market bonds?",
]

for i, prompt in enumerate(SAMPLE_PROMPTS):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    log(f"\nPrompt {i+1}: {prompt}")
    log(f"Response: {response[:200]}...")

In [ ]:
log("=== Saving Model ===")

# Save LoRA adapter
lora_path = "outputs/v1_optimized/lora_adapter"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
log(f"LoRA adapter saved to {lora_path}")

# Save training logs
logs_path = "results/training_logs/v1_optimized.json"
os.makedirs(os.path.dirname(logs_path), exist_ok=True)
with open(logs_path, "w") as f:
    json.dump(log_history, f, indent=2)
log(f"Training logs saved to {logs_path}")

log("\nTraining complete!")
log(f"Log file: {log_path}")
log("Download from Output tab after kernel completes.")